# tspgnn: multi-seed GPU training and evaluation

This notebook trains the distance-only GNN (`tspgnn/`) with several random seeds on a GPU,
then evaluates every seed with the repository's own benchmark harness (`tspbench`),
both greedy + 2-opt and the GNN-guided search.

**To run:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
Allow Google Drive access when asked.

Everything (training data, checkpoints, logs, benchmark results) is saved to
`MyDrive/tsp_gnn_solver/`. If the Colab session disconnects, just `Run all` again:
finished steps are skipped and training resumes from the last finished epoch.

The final table is written to `MyDrive/tsp_gnn_solver/<run>/SEEDS.md`.

In [ ]:
#@title Settings { display-mode: "form" }
#@markdown Git branch to run. Use `main` once the open PRs are merged.
BRANCH = "claude/colab-gpu-training-xtmix1"  #@param {type:"string"}
#@markdown Model seeds to train, comma-separated.
SEEDS = "0,1,2"  #@param {type:"string"}
#@markdown Training set (LKH-3 labelled, n = 20 to 100, mixed distance types). The CPU checkpoint used 40000.
TRAIN_NUM = 40000  #@param {type:"integer"}
EPOCHS = 10  #@param {type:"integer"}
HIDDEN = 64  #@param {type:"integer"}
LAYERS = 12  #@param {type:"integer"}
#@markdown Test instances per benchmark suite (Fu TSP500/1000 have 128).
EVAL_LIMIT = 32  #@param {type:"integer"}
#@markdown Guided-search budget in seconds per city (0.002 is what results/SEARCH.md uses).
TIME_PER_NODE = 0.002  #@param {type:"number"}
#@markdown Also evaluate the checkpoint shipped in the repo (trained on CPU) for comparison.
EVAL_SHIPPED_CHECKPOINT = True  #@param {type:"boolean"}
#@markdown Save to Google Drive, so a disconnected session can resume.
USE_DRIVE = True  #@param {type:"boolean"}
#@markdown Smoke test: a tiny model and data, about 5 minutes end to end even on CPU.
TINY = False  #@param {type:"boolean"}

In [ ]:
import os, subprocess, sys, json, glob, shutil

if TINY:
    TRAIN_NUM, EPOCHS, HIDDEN, LAYERS, EVAL_LIMIT, TIME_PER_NODE = 200, 2, 16, 2, 2, 0.0005
    SEEDS = "0,1"
SEED_LIST = [int(s) for s in SEEDS.split(",") if s.strip()]

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB and USE_DRIVE:
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/tsp_gnn_solver"
else:
    WORK = os.path.abspath("tsp_gnn_solver_work")
if TINY:
    WORK += "_tiny"
REPO = os.path.abspath("tsp_gnn_solver")
DATA = f"{WORK}/data/n{TRAIN_NUM}"
RUN = f"{WORK}/runs/h{HIDDEN}_l{LAYERS}_e{EPOCHS}_n{TRAIN_NUM}"
BENCH = f"{WORK}/bench_data"
EVAL = f"{RUN}/eval_t{TIME_PER_NODE:g}_lim{EVAL_LIMIT}"
os.makedirs(WORK, exist_ok=True)


def sh(cmd, cwd=None, env=None):
    """Run a shell command, stream its output, stop the notebook if it fails."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd or REPO, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, env={**os.environ, **(env or {})})
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")


print("work dir:", WORK)

## 1. Code and dependencies

In [ ]:
if os.path.isdir(f"{REPO}/.git"):
    sh(f"git fetch -q origin {BRANCH} && git checkout -q -B {BRANCH} FETCH_HEAD")
else:
    sh(f"git clone -q -b {BRANCH} https://github.com/pmandros/tsp_gnn_solver.git {REPO}", cwd=".")
sh("git log --oneline -1")
# Colab ships torch, numpy, scipy and numba. elkai is LKH-3 (labels and references).
sh(f"{sys.executable} -m pip install -q elkai==2.0.1 && {sys.executable} -m pip install -q -e . --no-deps")
sys.path.insert(0, REPO)

import torch
GPU = torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0) if GPU else "none, training runs on CPU (slow)")
print("CPU cores:", os.cpu_count())

## 2. Training data

Generated once and kept in Drive. LKH-3 labelling runs on the CPU
(about 25 minutes for 40,000 instances on Colab's 2 cores).

In [ ]:
if not os.path.exists(f"{DATA}/DONE"):
    os.makedirs(DATA, exist_ok=True)
    local = "/tmp/tspgnn_data"  # write locally, then copy: Drive is slow for many small writes
    shutil.rmtree(local, ignore_errors=True)
    sh(f"python scripts/make_data.py --out {local}/train.pt --num {TRAIN_NUM} --seed 1")
    sh(f"python scripts/make_data.py --out {local}/val.pt --num {max(20, min(600, TRAIN_NUM // 60))} "
       f"--seed 2 --store-d")
    for f in glob.glob(f"{local}/*.pt"):
        shutil.copy(f, DATA)
    open(f"{DATA}/DONE", "w").close()
print(sorted(os.listdir(DATA)))

## 3. Train one model per seed

Each seed is checkpointed every epoch. Re-running this cell resumes unfinished seeds
and skips finished ones. `best.pt` is the epoch with the lowest validation gap.

In [ ]:
def finished(run_dir):
    try:
        return len(json.load(open(f"{run_dir}/log.json"))) >= EPOCHS
    except (OSError, ValueError):
        return False


for seed in SEED_LIST:
    out = f"{RUN}/seed{seed}"
    if finished(out):
        print(f"seed {seed}: already trained")
        continue
    sh(f"python scripts/train.py --train '{DATA}/train*.pt' --val {DATA}/val.pt --out {out} "
       f"--epochs {EPOCHS} --hidden {HIDDEN} --layers {LAYERS} --seed {seed} --resume")

for seed in SEED_LIST:
    log = json.load(open(f"{RUN}/seed{seed}/log.json"))
    best = min(log, key=lambda r: r["val_greedy_gap"])
    print(f"seed {seed}: best val greedy gap {100 * best['val_greedy_gap']:.2f}% at epoch {best['epoch']}, "
          f"{log[-1]['time_s'] / 60:.0f} min")

## 4. Benchmark data and references

Fu et al. TSP500 and TSP1000 (LKH-3 references ship with the repo), Manhattan n = 500
(a distance type held out of training) and random non-metric n = 1000.
Reference tour lengths (LKH-3 / Concorde) for all of them ship in `data/refs/`.

In [ ]:
SUITES = ["tsp500", "tsp1000", "manhattan500:num=128", "nonmetric1000:num=128"]
if TINY:
    SUITES = ["tsp500", "nonmetric100:num=128"]

if not os.path.exists(f"{BENCH}/fu"):
    sh(f"python scripts/download_benchmarks.py --data-dir {BENCH} --no-tsplib")
os.makedirs(f"{BENCH}/refs", exist_ok=True)
for f in glob.glob(f"{REPO}/data/refs/*.json"):
    if not os.path.exists(f"{BENCH}/refs/{os.path.basename(f)}"):
        shutil.copy(f, f"{BENCH}/refs")
for suite in SUITES:
    # References for all these suites ship in data/refs; compute any that are missing with LKH-3.
    if not os.path.exists(f"{BENCH}/refs/{suite.replace(':', '_')}.json"):
        sh(f"python -m tspbench reference --suite {suite} --limit {EVAL_LIMIT} --solver lkh "
           f"--workers {os.cpu_count()} --data-dir {BENCH}")

## 5. Evaluate every seed

For each model: greedy + 2-opt, and the GNN-guided search with the same time budget per city.
The distance-guided search (the same search with edges ranked by distance, no model) runs once
as the baseline, on this machine, so the time budgets match.

In [ ]:
ENV = {"OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1", "NUMBA_NUM_THREADS": "1"}
SUITE_ARGS = " ".join(f"--suite {s}" for s in SUITES)
COMMON = f"--limit {EVAL_LIMIT} --seeds 0 --workers {os.cpu_count()} --data-dir {BENCH}"


def gnn_solvers(ckpt):
    return (f'--solver "callable:fn=tspgnn.api:solve,checkpoint={ckpt},name=gnn+2opt" '
            f'--solver "callable:fn=tspgnn.api:solve_search,guide=gnn,checkpoint={ckpt},'
            f'time_per_node={TIME_PER_NODE},name=gnn+search,stochastic=true"')


jobs = {f"seed{s}": gnn_solvers(f"{RUN}/seed{s}/best.pt") for s in SEED_LIST}
jobs["baseline"] = (f'--solver "callable:fn=tspgnn.api:solve_distance_greedy,name=dist+2opt" '
                    f'--solver "callable:fn=tspgnn.api:solve_search,guide=dist,'
                    f'time_per_node={TIME_PER_NODE},name=dist+search,stochastic=true"')
if EVAL_SHIPPED_CHECKPOINT:
    jobs["shipped_cpu"] = gnn_solvers(f"{REPO}/checkpoints/tspgnn.pt")

for name, solvers in jobs.items():
    out = f"{EVAL}/{name}"
    if os.path.exists(f"{out}/summary.csv"):
        print(f"{name}: already evaluated")
        continue
    sh(f"python -m tspbench run {SUITE_ARGS} {solvers} {COMMON} --out {out}", env=ENV)

## 6. Results across seeds

In [ ]:
sh(f"python scripts/seed_summary.py --eval-dir {EVAL} --out {RUN}/SEEDS.md")
from IPython.display import Markdown, display
display(Markdown(open(f"{RUN}/SEEDS.md").read()))
print("saved to", f"{RUN}/SEEDS.md")

Checkpoints are in `runs/<run>/seed*/best.pt` in the work folder. To use one elsewhere:
`tspgnn.api.solve(distance_matrix, checkpoint="…/best.pt")`.